In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

Libraries loaded.


In [2]:
df = pd.read_csv('diabetes_prediction_dataset.csv')
print("Shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'diabetes_prediction_dataset.csv'

In [ ]:
# PROJECT QUESTION
# ─────────────────────────────────────────────────────────────────
# Can we predict whether a patient has diabetes based on clinical
# and demographic features such as HbA1c level, blood glucose,
# BMI, age, hypertension, heart disease, and smoking history?
# Early prediction can help clinicians intervene before serious
# complications develop, improving patient outcomes globally.
# ─────────────────────────────────────────────────────────────────
print("Project question defined.")

In [ ]:
print("=== Shape ===")
print(df.shape)

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Duplicates ===")
print("Duplicate rows:", df.duplicated().sum())

print("\n=== Unique values in smoking_history ===")
print(df['smoking_history'].value_counts())

print("\n=== Target Distribution ===")
print(df['diabetes'].value_counts())
print(f"Diabetes rate: {df['diabetes'].mean()*100:.2f}%")

df.describe()

In [ ]:
# ── 1. Remove duplicate rows ─────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Removed {before - len(df)} duplicate rows.")

# ── 2. Replace 'No Info' in smoking_history with NaN then fill ───
# 'No Info' is not a valid category — it means data was not recorded
df['smoking_history'] = df['smoking_history'].replace('No Info', np.nan)
smoking_mode = df['smoking_history'].mode()[0]
df['smoking_history'].fillna(smoking_mode, inplace=True)
print(f"Replaced 'No Info' smoking_history with mode: '{smoking_mode}'")

# ── 3. Remove invalid gender entry ───────────────────────────────
invalid_gender = df[~df['gender'].isin(['Male', 'Female'])]
df = df[df['gender'].isin(['Male', 'Female'])]
print(f"Removed {len(invalid_gender)} rows with invalid gender.")

# ── 4. Fix BMI outliers — cap at 60 (medically unrealistic above) ─
bmi_outliers = (df['bmi'] > 60).sum()
df['bmi'] = df['bmi'].clip(upper=60)
print(f"Capped {bmi_outliers} BMI values above 60.")

# ── 5. Fix age — round to integer ────────────────────────────────
df['age'] = df['age'].astype(int)
print("Converted age to integer.")

# ── Final check ───────────────────────────────────────────────────
print("\nRemaining nulls:")
print(df.isnull().sum())
print("\nCleaned dataset shape:", df.shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Age distribution
axes[0].hist(df['age'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# BMI distribution
axes[1].hist(df['bmi'], bins=40, color='salmon', edgecolor='white')
axes[1].set_title('BMI Distribution')
axes[1].set_xlabel('BMI')

# Blood glucose distribution
axes[2].hist(df['blood_glucose_level'], bins=40,
             color='mediumseagreen', edgecolor='white')
axes[2].set_title('Blood Glucose Level Distribution')
axes[2].set_xlabel('Blood Glucose Level')

plt.tight_layout()
plt.savefig('plot1_univariate_numeric.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, ['gender', 'smoking_history', 'diabetes']):
    counts = df[col].value_counts()
    ax.bar(counts.index.astype(str), counts.values,
           color='mediumpurple', edgecolor='white')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('plot2_univariate_categorical.png', dpi=150)
plt.show()

print("\nClass imbalance check:")
print(df['diabetes'].value_counts())
print(f"Diabetes rate: {df['diabetes'].mean()*100:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x='diabetes', y='HbA1c_level',
            data=df, ax=axes[0], palette='Set2')
axes[0].set_title('HbA1c Level vs Diabetes')
axes[0].set_xticklabels(['No Diabetes', 'Diabetes'])
axes[0].set_xlabel('Diabetes')

sns.boxplot(x='diabetes', y='blood_glucose_level',
            data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Blood Glucose Level vs Diabetes')
axes[1].set_xticklabels(['No Diabetes', 'Diabetes'])
axes[1].set_xlabel('Diabetes')

plt.tight_layout()
plt.savefig('plot3_bivariate_boxplots.png', dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
numeric_df = df.select_dtypes(include=np.number)
corr = numeric_df.corr()

sns.heatmap(corr, annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.savefig('plot4_heatmap.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x='hypertension', hue='diabetes',
              data=df, ax=axes[0], palette='Set1')
axes[0].set_title('Hypertension vs Diabetes')
axes[0].set_xticklabels(['No Hypertension', 'Hypertension'])
axes[0].legend(['No Diabetes', 'Diabetes'])

sns.countplot(x='heart_disease', hue='diabetes',
              data=df, ax=axes[1], palette='Set1')
axes[1].set_title('Heart Disease vs Diabetes')
axes[1].set_xticklabels(['No Heart Disease', 'Heart Disease'])
axes[1].legend(['No Diabetes', 'Diabetes'])

plt.tight_layout()
plt.savefig('plot5_countplots.png', dpi=150)
plt.show()

In [ ]:
def get_age_group(age):
    """Converts raw age into a clinical risk group label."""
    if age < 30:   return 'Young'
    elif age < 45: return 'Middle Aged'
    elif age < 60: return 'Pre-Senior'
    else:          return 'Senior'

def get_bmi_category(bmi):
    """Converts raw BMI into a clinical weight category label."""
    if bmi < 18.5: return 'Underweight'
    elif bmi < 25: return 'Normal'
    elif bmi < 30: return 'Overweight'
    else:          return 'Obese'

def get_glucose_risk(glucose):
    """
    Classifies blood glucose level into clinical risk categories
    based on standard medical thresholds.

    Parameters:
        glucose : float blood glucose reading
    Returns:
        string risk label
    """
    if glucose < 100:   return 'Normal'
    elif glucose < 126: return 'Pre-Diabetic'
    else:               return 'Diabetic Range'

def get_risk_score(row):
    """
    Combines 4 binary risk factors into a single composite score.

    Parameters:
        row : one row of the dataframe
    Returns:
        integer score from 0 to 4
    """
    score = 0
    if row['hypertension'] == 1:          score += 1
    if row['heart_disease'] == 1:         score += 1
    if row['bmi'] >= 30:                  score += 1
    if row['HbA1c_level'] >= 6.5:         score += 1
    return score

# ── Apply all new features ──────────────────────────────────────
df['age_group']     = df['age'].apply(get_age_group)
df['bmi_category']  = df['bmi'].apply(get_bmi_category)
df['glucose_risk']  = df['blood_glucose_level'].apply(get_glucose_risk)
df['risk_score']    = df.apply(get_risk_score, axis=1)

print("New features added:")
print(df[['age_group', 'bmi_category',
          'glucose_risk', 'risk_score']].head(10))
print("\nRisk score distribution:")
print(df['risk_score'].value_counts().sort_index())

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler

# ── Encode all categorical columns ──────────────────────────────
df_encoded = df.copy()
le = LabelEncoder()

cat_cols = df_encoded.select_dtypes(include='object').columns
for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
print("Encoded columns:", list(cat_cols))

# ── Drop any remaining nulls ─────────────────────────────────────
before = len(df_encoded)
df_encoded.dropna(inplace=True)
print(f"Dropped {before - len(df_encoded)} rows with nulls.")

# ── Split features and target ────────────────────────────────────
X = df_encoded.drop(columns=['diabetes'])
y = df_encoded['diabetes']

print(f"\nNaNs in X before scaling: {X.isnull().sum().sum()}")

# ── Scale to 0-1 range ───────────────────────────────────────────
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

print(f"NaNs in X after scaling: {X_scaled.isnull().sum().sum()}")

# ── Feature Selection: SelectKBest chi2 (Filter Method) ─────────
# chi2 scores each feature's statistical relationship with target
# higher score = more relevant to predicting diabetes
selector = SelectKBest(chi2, k=8)
selector.fit(X_scaled, y)

selected_features = X.columns[selector.get_support()]
print("\nTop 8 selected features:")
print(list(selected_features))

X_final = X_scaled[selected_features]
print("\nFinal feature matrix shape:", X_final.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# ── WHAT IS VALIDATION AND WHY IT MATTERS ──────────────────────
# Validation means testing your model on data it has NEVER seen
# during training. Without it, the model might memorize training
# data (overfitting) and fail on real new patients.
# We split 80% for training and 20% for testing to simulate
# how the model performs on unseen real-world data.
# ───────────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # keeps diabetes ratio equal in both splits
)

print("Before SMOTE:")
print(y_train.value_counts())

# SMOTE fixes class imbalance (~8.5% diabetes rate)
# by generating synthetic diabetic patient samples
sm = SMOTE(random_state=42)
X_train, y_train = sm.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(pd.Series(y_train).value_counts())
print(f"\nTraining samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ── WHAT IS PARAMETER TUNING AND WHY IT MATTERS ────────────────
# Every algorithm has hyperparameters that control how it learns.
# Default settings are rarely optimal for a specific dataset.
# Tuning searches for the best combination to maximize performance
# while preventing underfitting and overfitting.
# ───────────────────────────────────────────────────────────────

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    """
    Trains a model and returns its performance metrics as a dictionary.

    Parameters:
        name    : string label for the model
        model   : sklearn model object
        X_train : training features
        X_test  : testing features
        y_train : training labels
        y_test  : testing labels

    Returns:
        dict with model name and 4 metric scores
    """
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    return {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred),  3),
        'Precision': round(precision_score(y_test, y_pred), 3),
        'Recall':    round(recall_score(y_test, y_pred),    3),
        'F1 Score':  round(f1_score(y_test, y_pred),        3)
    }

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest':       RandomForestClassifier(random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=42)
}

print("Training all 3 models...")
results = [
    evaluate_model(name, model, X_train, X_test, y_train, y_test)
    for name, model in models.items()
]

results_df = pd.DataFrame(results)
print("\n=== Model Comparison ===")
print(results_df.to_string(index=False))

In [ ]:
def plot_model_comparison(results_df, save_path='plot6_model_comparison.png'):
    """
    Plots a grouped bar chart comparing all models across 4 metrics.

    Parameters:
        results_df : DataFrame with Model, Accuracy, Precision, Recall, F1
        save_path  : filename to save the figure
    """
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
    colors  = ['steelblue', 'salmon', 'mediumseagreen', 'mediumpurple']
    x       = np.arange(len(results_df))
    width   = 0.2

    fig, ax = plt.subplots(figsize=(10, 5))

    for i, (metric, color) in enumerate(zip(metrics, colors)):
        offset = (i - 1.5) * width
        ax.bar(x + offset, results_df[metric], width,
               label=metric, color=color)

    ax.set_xticks(x)
    ax.set_xticklabels(results_df['Model'])
    ax.set_ylim(0, 1.1)
    ax.set_title('Model Comparison — All Metrics')
    ax.set_ylabel('Score')
    ax.legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()

plot_model_comparison(results_df)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [10, 20],
    'min_samples_split': [2, 5],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print("Running GridSearchCV... ~2-3 minutes.")
grid_search.fit(X_train, y_train)

print("\nBest Parameters Found:")
print(grid_search.best_params_)
print(f"Best Cross-Val F1 Score: {grid_search.best_score_:.3f}")

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                              ConfusionMatrixDisplay)

def print_final_metrics(y_test, y_pred, model_name='Model'):
    """
    Prints a clean summary of precision, recall, F1, and accuracy.

    Parameters:
        y_test     : true labels
        y_pred     : predicted labels
        model_name : string label shown in output header
    """
    print(f"\n=== Final Metrics: {model_name} ===")
    print(f"Precision : {precision_score(y_test, y_pred):.3f}")
    print(f"Recall    : {recall_score(y_test, y_pred):.3f}")
    print(f"F1 Score  : {f1_score(y_test, y_pred):.3f}")
    print(f"Accuracy  : {accuracy_score(y_test, y_pred):.3f}")

# Use the best model found by GridSearchCV
best_model  = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

# ── Classification Report ────────────────────────────────────────
print("=== Classification Report ===")
print(classification_report(y_test, y_pred_best,
      target_names=['No Diabetes', 'Diabetes']))

# ── Confusion Matrix ─────────────────────────────────────────────
cm   = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['No Diabetes', 'Diabetes'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Tuned Random Forest')
plt.tight_layout()
plt.savefig('plot7_confusion_matrix.png', dpi=150)
plt.show()

# ── Final metrics ────────────────────────────────────────────────
print_final_metrics(y_test, y_pred_best, model_name='Tuned Random Forest')

In [ ]:
def plot_feature_importance(model, feature_names,
                             save_path='plot8_feature_importance.png'):
    """
    Plots a horizontal bar chart of feature importances.

    Parameters:
        model         : trained sklearn model with feature_importances_
        feature_names : list or Index of feature column names
        save_path     : filename to save the figure
    """
    feat_df = pd.DataFrame({
        'Feature':    feature_names,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=False)

    plt.figure(figsize=(10, 5))
    sns.barplot(x='Importance', y='Feature',
                data=feat_df, palette='viridis')
    plt.title('Feature Importances — Tuned Random Forest')
    plt.tight_layout()
    plt.savefig('plot8_feature_importance.png', dpi=150)
    plt.show()

plot_feature_importance(best_model, X_final.columns)

In [ ]:
print("""
=== Project Conclusions ===

Question: Can we predict diabetes risk from patient clinical data?

Answer: YES — the tuned Random Forest model successfully identifies
diabetic patients from clinical and demographic features.

Key findings from EDA and modeling:
- HbA1c level and blood glucose are the strongest predictors
- Obese patients (BMI >= 30) show significantly higher diabetes rates
- Hypertension and heart disease both correlate with diabetes risk
- The engineered risk_score feature ranked highly, validating
  the feature engineering step
- Age over 45 is a strong non-modifiable risk factor

The model can assist clinicians in flagging high-risk patients
for early intervention before complications develop.
""")

In [ ]:
import joblib
import json

joblib.dump(best_model, 'diabetes_model.pkl')
print("Saved: diabetes_model.pkl")

joblib.dump(scaler, 'scaler.pkl')
print("Saved: scaler.pkl")

joblib.dump(selector, 'selector.pkl')
print("Saved: selector.pkl")

with open('selected_features.json', 'w') as f:
    json.dump(list(selected_features), f)
print("Saved: selected_features.json")

print("\nAll artifacts saved!")

In [ ]:
from google.colab import files

files.download('diabetes_model.pkl')
files.download('scaler.pkl')
files.download('selector.pkl')
files.download('selected_features.json')

print("Check your Downloads folder!")